In [ ]:
import numpy as np
import pandas as pd
from pandas.plotting import parallel_coordinates

import shap

from scipy.sparse import diags, csr_matrix, hstack, vstack

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.pipeline import Pipeline, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt

# Dataset Overview: CiteULike
For this seminar, we will focus on the CiteULike dataset, a widely used benchmark in the domain of recommender systems. CiteULike is a web service that allows users to bookmark and tag scientific articles, providing a rich source of interaction data between users and scholarly publications.

### Key Characteristics:
- Users : The dataset includes interactions from 5,551 unique users , representing a diverse range of research interests.
- Items (Articles) : There are 16,980 scientific articles available in the dataset. Each article is described by its title and abstract , which are concatenated together to form a comprehensive textual representation.
- Textual Representation : To simplify the modeling process, we have already preprocessed the textual descriptions of the articles. Specifically, we extracted the 8,000 most important word tokens based on their frequency and relevance. These - tokens serve as the feature space for our model, enabling us to capture meaningful semantic information about the articles.

This dataset provides an ideal foundation for exploring content-based recommendation techniques, as it combines user preferences with detailed item descriptions in a structured format.

### Predicting Article Interactions Using Textual Descriptions
Our primary goal is to develop a content-based model that leverages the textual descriptions of articles to predict the likelihood or intensity of interactions with articles. In this context, "interactions" could refer to actions such as bookmarking, tagging, or reading an article. The purpose of this seminar is to introduce you to simple content-based models and analyze how content-based features impact the model's ability to make accurate predictions. We will consider more complex and personalized models later in this course.

Each row in the dataframe corresponds to a specific user, and the columns represent the articles they have interacted with. The interactions are encoded as strings containing article identifiers, where each string lists the articles associated with a particular user.

In [ ]:
history_raw = pd.read_csv('./citeulike-a/users.dat', engine='python', names=['itemid'])
history_raw.index.name = 'userid'
history_raw

In [ ]:
items_raw = pd.read_csv('./citeulike-a/raw-data.csv')
items_raw.loc[:, 'doc.id'] = np.arange(len(items_raw))
items_raw

The preprocessed titles and abstracts, described by 8000 features: number of features in article, followed by (feature_id:number of features).

In [ ]:
preprocessed_items_raw = pd.read_csv('./citeulike-a/mult.dat', engine='python', names=['features'])
preprocessed_items_raw.index.name = 'itemid'
preprocessed_items_raw

Let's prepare the interactions and features data in csr format.

In [ ]:
history_length = ...

user_idx = ...
item_idx = ...

interactions_matrix = csr_matrix(
    (np.ones_like(item_idx), (user_idx, item_idx)),
    shape=(len(history_raw), len(items_raw))
)

In [ ]:
splitted_features = preprocessed_items_raw['features'].str.split(' ')

features_length = splitted_features.apply(lambda x: int(x[0])).values

feature_idx    = np.concatenate(splitted_features.apply(lambda x: [int(el.split(':')[0]) for el in x[1:]]).values)
multiplicities = np.concatenate(splitted_features.apply(lambda x: [int(el.split(':')[1]) for el in x[1:]]).values)
item_idx       = np.repeat(np.arange(len(items_raw)), features_length)

feature_matrix = csr_matrix(
    (multiplicities, (item_idx, feature_idx)),
    shape=(len(preprocessed_items_raw), 8000)
)

The matplotlib.pyplot.spy function allows to look at the sparsity pattern of a matrix.

In [ ]:
plt.figure(figsize=(16, 9))
plt.spy(interactions_matrix, markersize=0.05)
plt.ylabel('User')
plt.xlabel('Item')
plt.show()

In [ ]:
plt.figure(figsize=(9, 16))
plt.spy(feature_matrix, markersize=0.01)
plt.xlabel('Feature')
plt.ylabel('Item')
plt.show()

In [ ]:
plt.semilogy(np.sort(feature_matrix.astype(bool).astype(int).sum(axis=0).A.squeeze()))
plt.xlabel('Feature')
plt.ylabel('Log occurence')
plt.show()

In [ ]:
plt.semilogy(np.sort(interactions_matrix.sum(axis=0).A.squeeze()))
plt.xlabel('Item')
plt.ylabel('Log occurence')
plt.show()

In [ ]:
vocab = pd.read_csv('./citeulike-a/vocabulary.dat', engine='python', names=['name'])
vocab

Least occuring features

In [ ]:
features_sorted_idx = ...

vocab.iloc[features_sorted_idx[:10]]

Most occuring features

In [ ]:
vocab.iloc[features_sorted_idx[-10:][::-1]]

Let's predict the popularity of a newly introduced article - how many users are going to interact with it. We can get the number of interactions by summing the interactions matrix across user dimension.

In [ ]:
y = ...
X = ...

We will split the data randomly, leaving 10\% of items for testing.

In [ ]:
X_train_idx, X_test_idx, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=2025)

X_train = feature_matrix[X_train_idx.index]
X_test = feature_matrix[X_test_idx.index]

Make sure there are no items without any features.

In [ ]:
print(f"train items w/o features: {(X_train.sum(axis=1) == 0).sum()}")
print(f"test items w/o features: {(X_test.sum(axis=1) == 0).sum()}")

Let's also look at the feature coverage

In [ ]:
print(f"{(X_train.sum(axis=0) == 0).sum()} features are not presented in train")
print(f"{(X_test.sum(axis=0) == 0).sum()} features are not presented in test")

In [ ]:
def evaluate(model, X, y):
    '''
    Computes RMSE and MAE scores
    '''
    y_pred = model.predict(X)
    results = pd.DataFrame.from_dict(
        {
            'index':[model.__class__.__name__],
            'columns': ['RMSE', 'MAE'],
            'data': [[..., ...]],
            'index_names':[None],
            'column_names':[None]
            },
        orient='tight'
    )
    return results

We will consider the Linear Regression model with its modifications. The main assumption is that the number of interactions linearly depends on the features of item:

$$
\bold{y}=\bold{X}\bold{w} +\bold{\epsilon}
$$
For a single data point k, the prediction is given by:
$$
y_k=\bold{x}_k^\top\bold{w} +\epsilon_k
$$

The optimization task is:

$$
\|\bold{y}-\bold{X}\bold{w}\|_2^2\rightarrow \min
$$

We can also add different regularizations - $l_1$ and $l_2$ norm of model's weights $\bold{w}$ with coefficients, resulting in Lasso($l_1$), Ridge($l_2$) and ElasticNet($l_1$+$l_2$) models.

In [ ]:
model_zoo = [...]

results_no_preproc = pd.DataFrame()
for model in model_zoo:
    model.fit(X_train, y_train)
    results_no_preproc = pd.concat([results_no_preproc, evaluate(model, X_test, y_test)], axis=0)

In [ ]:
results_no_preproc.round(2)

In [ ]:
for model in model_zoo:
    print(model.__class__.__name__)
    print(f'w0={model.intercept_:.2f}, ||w||={np.linalg.norm(model.coef_):.2f}')

Lets see the terms importance via model weights magnitude

In [ ]:
vocab.iloc[...]

In [ ]:
feature_matrix.toarray().max(axis=0)

# Interpreting Models with SHAP

To enhance the interpretability of our Linear Regression model and its regularized variants (Ridge, Lasso, Elastic Net), we will utilize **SHAP** (**SH**apley **A**dditive ex**P**lanations). SHAP is a game-theoretic approach for explaining the output of machine learning models, providing insights into how each feature contributes to individual predictions.

What Are SHAP Values?
SHAP values are based on the concept of Shapley values from cooperative game theory. They assign a value to each feature for a given prediction, representing the contribution of that feature to the final output. Key properties of SHAP values include:

Local Accuracy : The sum of the SHAP values for all features plus the base value equals the model's output for that instance.
Consistency : If a feature contributes more to the prediction when compared to another feature, its SHAP value will be higher.
Missingness : Features that are missing or irrelevant have no impact on the prediction.


For further guidance and inspiration, refer to the official SHAP documentation and several blogposts:
- https://shap.readthedocs.io/en/latest/index.html
- https://www.deepchecks.com/a-comprehensive-guide-into-shap-shapley-additive-explanations-values/
- https://medium.com/towards-data-science/a-novel-approach-to-feature-importance-shapley-additive-explanations-d18af30fc21b



In [ ]:
explainer = shap.explainers.Linear(model_zoo[0], X_train, feature_names=vocab['name'])
shap_values = explainer(X_train)

shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.bar(shap_values, max_display=10)

In [ ]:
index = 1000
X_train_idx.loc[index]

In [ ]:
X_train_idx.loc[index]['raw.abstract'].split('.') + X_train_idx.loc[index]['raw.title'].split('.')

In [ ]:
vocab.iloc[X_train[np.where(X_train_idx['doc.id'] == index)[0][0], :].indices].values.squeeze()

We can also investigate how each particular feature impacts the output. Let's take our article with index 1000 and plot the 'waterfall'. In lower left corner the value $E[f(X)]=12.689$ is the averaged output on training data. In upper right $f(x)=16.807$ is the output for this particular article. The feature 'machine' has a positive effects on model's output, while 'irrelevant', 'selecting' and 'close' have negative impact.

In [ ]:
shap.plots.waterfall(shap_values[np.where(X_train_idx['doc.id'] == index)[0][0]], max_display=10)

In [ ]:
shap.plots.scatter(shap_values[:, ["selection", "features"]])

In [ ]:
shap.summary_plot(shap_values, features=X_train, feature_names=vocab.values.squeeze(), plot_type='bar', max_display=10)

# Feature preprocessing

The next critical step in building our content-based recommendation system is feature preprocessing, where we transform the raw document-term matrix into a more meaningful representation that enhances the model's ability to capture relevant patterns. Two widely used techniques for this purpose are TF-IDF (Term Frequency-Inverse Document Frequency) and feature scaling . While these methods provide robust starting points, it's important to recognize that feature preprocessing is as much an art as it is a science—requiring creativity, domain knowledge, and iterative experimentation.

### TF-IDF
The document-term matrix represents each article as a vector of word frequencies. However, raw frequency counts can be misleading because common words (e.g., "the," "and") appear frequently across all documents but carry little discriminative power. To address this issue, TF-IDF adjusts term weights by considering both their frequency within a document and their rarity across the corpus.

The TF-IDF weight for a term t in document d is calculated as:

TF-IDF(t,d)=TF(t,d)×IDF(t)

$$
\text{TF}(t, d) = \frac{f_{t, d}}{\sum_{t'\in d}f_{t', d}}
$$
$$
\text{IDF}(t, D) = \log\left( \frac{N}{1 + |\{d\in D: t\in d\}|} \right) + 1
$$

In [ ]:
class TFIDF_Transformer(TransformerMixin):
    '''
    Apply TF-IDF transformation to the document-term matrix
    '''

    def fit(self, X, y=None, **fit_params):
        self.corpus = X.copy()
        idf_scores = self.corpus.astype(bool).sum(axis=0).A.squeeze()
        self.idf = diags(np.log(self.corpus.shape[0] / (1.0 + idf_scores)) + 1.0)
        return self
    
    def transform(self, X, y=None, **fit_params):
        return ...

In [ ]:
class DenseTransformer(TransformerMixin):
    """
    Convert sparse matrix to dense np array to apply standard scaler with mean.
    """

    def fit(self, X, y=None, **fit_params):
        return self

    def transform(self, X, y=None, **fit_params):
        return X.toarray()

### Feature Scaling
After applying TF-IDF, the resulting feature values may span several orders of magnitude, depending on the dataset. This can lead to numerical instability or bias in models that rely on distance metrics. To mitigate this issue, feature scaling standardizes the range of features, ensuring that no single feature dominates the others due to its scale.



In [ ]:
word_vectorizer = Pipeline([
    ("tfidf", TFIDF_Transformer()), 
    ("dense", DenseTransformer()),
    # ("scaler", StandardScaler())
])

word_vectorizer

In [ ]:
model_zoo = [LinearRegression(), Ridge(), Lasso(), ElasticNet()]

feature_matrix_train = word_vectorizer.fit_transform(X_train)
results_tfidf = pd.DataFrame()
for model in model_zoo:
    model.fit(feature_matrix_train, y_train)
    results_tfidf = pd.concat([results_tfidf, evaluate(model, word_vectorizer.transform(X_test), y_test)], axis=0)

In [ ]:
results_tfidf.round(2)

In [ ]:
results_no_preproc.round(2)

In [ ]:
model_zoo[0].coef_

In [ ]:
vocab.iloc[np.abs(model_zoo[0].coef_).argsort()[::-1][:10]]

In [ ]:
for model in model_zoo:
    print(model.__class__.__name__)
    print(f'w0={model.intercept_:.2f}, ||w||={np.linalg.norm(model.coef_):.2f}')

In [ ]:
plt.figure(figsize=(10, 10))
weights_df = pd.DataFrame.from_dict({model.__class__.__name__: np.argsort(-model.coef_) for model in model_zoo})
weights_df['names'] = vocab.iloc[weights_df['Lasso']]


parallel_coordinates(
    weights_df,
    class_column='names',
    cols=['ElasticNet', 'Lasso', 'Ridge', 'LinearRegression'],
    linewidth=1
)

plt.ylabel('Position')
plt.gca().legend_.remove()

# Binary classification

In [ ]:
X = feature_matrix.copy()
y_binary = ...

X_train, X_test, y_bin_train, y_bin_test = train_test_split(X, y_binary, test_size=0.1, random_state=2025)

In [ ]:
word_vectorizer = Pipeline([
    ("tfidf", TFIDF_Transformer()),
])

feature_matrix_train = word_vectorizer.fit_transform(X_train)

model = LogisticRegression()
model.fit(feature_matrix_train, y_bin_train)

In [ ]:
np.round(
    accuracy_score(
        y_true=y_bin_test.astype(int),
        y_pred=model.predict(word_vectorizer.transform(X_test))
    ),
2).item()

In [ ]:
np.round(
    accuracy_score(
        y_true=y_bin_test.astype(int),
        y_pred=(np.random.rand(len(y_bin_test)) >= 0.5).astype(int)
    ),
2).item()

We managed to beat the random baseline! 

# More personalization

In [ ]:
np.min(history_length)

Lets take first 10 items as items than describes a user

In [ ]:
X = []
y = []

for u, items in history_raw['itemid'].items():
    inference_items = items[:10]
    positive_items = items[10:]

    train_items = positive_items

    user_feat = feature_matrix[inference_items].mean(axis=0).A.squeeze()

    X.append(
        csr_matrix(
            np.hstack([
                np.tile(user_feat, (len(train_items), 1)),
                feature_matrix[train_items].toarray()
            ])
        )
    )
    y.append(np.ones(len(positive_items)))

X = vstack(X)
y = np.concatenate(y)

We can check how much space of RAM we consume with X matrix

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=True)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
np.round(
    accuracy_score(
        y_true=y_test,
        y_pred=model.predict(X_test)
    ),
2).item()

In [ ]:
np.round(
    accuracy_score(
        y_true=y_test,
        y_pred=(np.random.rand(len(y_test)) >= 0.5).astype(int)
    ),
2).item()